# Trade Operations Reconciliation & Exception Monitoring

This notebook adds a lightweight **Trading Operations** layer to the existing FX & CFD trading analytics project.

The purpose is to simulate how an operations analyst would reconcile the trading lifecycle:

1. start from internal trade records generated from the strategy workflow;
2. compare them against broker confirmations;
3. compare confirmed trades against settlement records;
4. flag operational exceptions using tolerance thresholds;
5. export structured exception reports and monitoring scorecards.


## 1. Import libraries and define project paths

This step imports the core libraries and creates the folders used by this module.

- `data/trade_operations/` stores the simulated trade lifecycle datasets.
- `trade_operations_outputs/` stores the exception reports, summaries, and scorecards.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set a fixed seed so the simulated trade operations records are reproducible.
np.random.seed(42)

# Define project paths relative to the notebook folder.
# The notebook is expected to run from the notebooks/ directory.
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
TRADE_OPS_DIR = DATA_DIR / "trade_operations"
OUTPUT_DIR = PROJECT_ROOT / "trade_operations_outputs"

# Create output folders if they do not already exist.
TRADE_OPS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Trade operations data directory:", TRADE_OPS_DIR)
print("Output directory:", OUTPUT_DIR)


Trade operations data directory: ../data/trade_operations
Output directory: ../trade_operations_outputs


## 2. Load the existing trading performance dataset

Instead of creating unrelated sample data, this module uses the existing `trade_performance.csv` output from the main trading analytics workflow.

Rows with non-zero turnover are treated as simulated trade events. This keeps the operations module connected to the original trading strategy, market data, transaction cost, and PnL analysis.


In [2]:
# Load the trading performance dataset created by the main project workflow.
trade_perf = pd.read_csv(DATA_DIR / "trade_performance.csv")

# Convert the Date column into datetime format for trade lifecycle calculations.
trade_perf["Date"] = pd.to_datetime(trade_perf["Date"])

# Display the dataset size and a preview for validation.
print("Trading performance dataset shape:", trade_perf.shape)
trade_perf.head()


Trading performance dataset shape: (6117, 27)


,Date,Open,High,Low,Close,Volume,asset,ticker,daily_return,log_return,...,spread_cost_rate,transaction_cost,slippage_cost,total_cost,strategy_return_after_cost,cumulative_return_before_cost,cumulative_return_after_cost,equity_curve_after_cost,running_peak_after_cost,drawdown_after_cost
0,2021-03-11,0.773140,0.778420,0.772440,0.773200,0,AUDUSD,AUDUSD=X,0.001836,0.001834,...,0.00007,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000
1,2021-03-12,0.778510,0.780100,0.772670,0.778762,0,AUDUSD,AUDUSD=X,0.007193,0.007167,...,0.00007,0.00007,0.000764,0.000834,0.006359,0.007193,0.006359,1.006359,1.006359,0.000000
2,2021-03-15,0.776337,0.777600,0.770677,0.776300,0,AUDUSD,AUDUSD=X,-0.003161,-0.003166,...,0.00007,0.00000,0.000000,0.000000,-0.003161,0.004009,0.003178,1.003178,1.006359,-0.003161
3,2021-03-16,0.775000,0.775700,0.771350,0.775014,0,AUDUSD,AUDUSD=X,-0.001657,-0.001658,...,0.00007,0.00000,0.000000,0.000000,-0.001657,0.002345,0.001515,1.001515,1.006359,-0.004813
4,2021-03-17,0.774659,0.774713,0.770620,0.774557,0,AUDUSD,AUDUSD=X,-0.000589,-0.000589,...,0.00007,0.00000,0.000000,0.000000,-0.000589,0.001755,0.000926,1.000926,1.006359,-0.005399


## 3. Create simulated internal trade records

This step creates the internal trade blotter, which represents trades recorded by the internal trading system.

The table includes:

- unique `trade_id`;
- trade date and asset;
- buy/sell direction;
- trade quantity;
- execution price;
- notional value;
- expected PnL after strategy return and cost adjustment.

This is the baseline dataset used for reconciliation.


In [3]:
# Select rows where the strategy generated a trade.
# In this project, turnover > 0 indicates that the position changed and a trade occurred.
trade_events = trade_perf[trade_perf["turnover"] > 0].copy()

# Keep the module lightweight and easy to inspect on GitHub.
trade_events = (
    trade_events
    .sort_values(["Date", "asset"])
    .head(120)
    .reset_index(drop=True)
)

# Build a simulated internal trade blotter from strategy-generated trade events.
internal_trades = pd.DataFrame({
    "trade_id": [f"T{str(i + 1).zfill(5)}" for i in range(len(trade_events))],
    "trade_date": trade_events["Date"],
    "asset": trade_events["asset"],
    "side": np.where(trade_events["position"] > 0, "BUY", "SELL"),
    "quantity": np.random.randint(10, 200, size=len(trade_events)) * 100,
    "execution_price": trade_events["Close"].round(5),
    "strategy_return_after_cost": trade_events["strategy_return_after_cost"],
    "total_cost": trade_events["total_cost"],
    "trader": np.random.choice(["Desk_A", "Desk_B", "Desk_C"], size=len(trade_events)),
    "strategy": "MA_Crossover"
})

# Calculate notional value for each trade.
internal_trades["notional"] = (
    internal_trades["quantity"] * internal_trades["execution_price"]
).round(2)

# Estimate expected PnL using after-cost strategy return and transaction cost fields.
# This creates a simplified PnL reference point for later reconciliation.
internal_trades["expected_pnl"] = (
    internal_trades["notional"] * internal_trades["strategy_return_after_cost"]
    - internal_trades["notional"] * internal_trades["total_cost"]
).round(2)

# Save the internal trade blotter for downstream reconciliation.
internal_trades.to_csv(TRADE_OPS_DIR / "internal_trades.csv", index=False)

print("Internal trades generated:", internal_trades.shape)
internal_trades.head()


Internal trades generated: (120, 12)


,trade_id,trade_date,asset,side,quantity,execution_price,strategy_return_after_cost,total_cost,trader,strategy,notional,expected_pnl
0,T00001,2021-03-12,AUDUSD,BUY,11200,0.77876,0.006359,0.000834,Desk_A,MA_Crossover,8722.11,48.19
1,T00002,2021-03-17,SP500,BUY,18900,3974.12012,0.001456,0.001424,Desk_C,MA_Crossover,75110870.27,2409.66
2,T00003,2021-03-29,AUDUSD,SELL,10200,0.76374,-0.000739,0.000739,Desk_C,MA_Crossover,7790.15,-11.51
3,T00004,2021-04-14,NASDAQ100,BUY,2400,13803.91016,-0.014657,0.001603,Desk_B,MA_Crossover,33129384.38,-538662.40
4,T00005,2021-04-26,GOLD,BUY,11600,1779.19995,0.000103,0.001135,Desk_A,MA_Crossover,20638719.42,-21282.93


## 4. Create simulated broker confirmations

This step creates the external broker confirmation file.

In a real trading environment, operations teams compare internal trade records against confirmations received from brokers or counterparties. To simulate realistic trade breaks, this notebook intentionally introduces controlled exceptions:

- missing confirmations;
- quantity mismatches;
- price discrepancies;
- reported PnL variances.



In [4]:
# Start from internal trades and keep only fields that a broker confirmation would normally contain.
broker_confirmations = internal_trades[[
    "trade_id", "trade_date", "asset", "quantity", 
    "execution_price", "expected_pnl"
]].copy()

# Rename fields to represent externally confirmed values.
broker_confirmations = broker_confirmations.rename(columns={
    "trade_date": "confirmation_date",
    "quantity": "confirmed_quantity",
    "execution_price": "confirmed_price",
    "expected_pnl": "reported_pnl"
})

# Simulate confirmations arriving on trade date or one day later.
broker_confirmations["confirmation_date"] = (
    pd.to_datetime(broker_confirmations["confirmation_date"])
    + pd.to_timedelta(
        np.random.choice([0, 1], size=len(broker_confirmations)),
        unit="D"
    )
)

# Add small normal price noise to represent normal broker/system rounding differences.
broker_confirmations["confirmed_price"] = (
    broker_confirmations["confirmed_price"]
    * (1 + np.random.normal(0, 0.0002, size=len(broker_confirmations)))
).round(5)

# Add small normal PnL noise to represent minor reporting differences.
broker_confirmations["reported_pnl"] = (
    broker_confirmations["reported_pnl"]
    + np.random.normal(0, 5, size=len(broker_confirmations))
).round(2)

# Select trade IDs where controlled exceptions will be introduced.
missing_confirmation_ids = broker_confirmations.sample(5, random_state=1)["trade_id"].tolist()
quantity_mismatch_ids = broker_confirmations.sample(6, random_state=2)["trade_id"].tolist()
price_mismatch_ids = broker_confirmations.sample(6, random_state=3)["trade_id"].tolist()
pnl_variance_ids = broker_confirmations.sample(6, random_state=4)["trade_id"].tolist()

# Remove selected rows to simulate missing broker confirmations.
broker_confirmations = broker_confirmations[
    ~broker_confirmations["trade_id"].isin(missing_confirmation_ids)
].copy()

# Increase confirmed quantity for selected trades to simulate quantity mismatches.
broker_confirmations.loc[
    broker_confirmations["trade_id"].isin(quantity_mismatch_ids),
    "confirmed_quantity"
] = broker_confirmations["confirmed_quantity"] + 100

# Increase confirmed price for selected trades to simulate material price discrepancies.
broker_confirmations.loc[
    broker_confirmations["trade_id"].isin(price_mismatch_ids),
    "confirmed_price"
] = (broker_confirmations["confirmed_price"] * 1.01).round(5)

# Increase reported PnL for selected trades to simulate unexplained PnL variances.
broker_confirmations.loc[
    broker_confirmations["trade_id"].isin(pnl_variance_ids),
    "reported_pnl"
] = broker_confirmations["reported_pnl"] + 250

# Add a broker status field for operational context.
broker_confirmations["broker_status"] = "CONFIRMED"

# Save broker confirmations for reconciliation.
broker_confirmations.to_csv(TRADE_OPS_DIR / "broker_confirmations.csv", index=False)

print("Broker confirmations generated:", broker_confirmations.shape)
broker_confirmations.head()


Broker confirmations generated: (115, 7)


,trade_id,confirmation_date,asset,confirmed_quantity,confirmed_price,reported_pnl,broker_status
0,T00001,2021-03-13,AUDUSD,11200,0.77872,49.59,CONFIRMED
1,T00002,2021-03-18,SP500,18900,3973.52103,2406.55,CONFIRMED
2,T00003,2021-03-29,AUDUSD,10300,0.76360,237.45,CONFIRMED
3,T00004,2021-04-14,NASDAQ100,2400,13801.65789,-538664.87,CONFIRMED
4,T00005,2021-04-27,GOLD,11600,1779.17251,-21285.88,CONFIRMED


## 5. Create simulated settlement records

This step creates the settlement file, representing whether confirmed trades were settled successfully.

The notebook intentionally introduces:

- pending settlements;
- failed settlements;
- delayed settlements.



In [5]:
# Use confirmed trades as the starting point for settlement records.
settlement_records = broker_confirmations[[
    "trade_id", "confirmation_date", "asset", "confirmed_quantity", "confirmed_price"
]].copy()

# Simulate settlement date as 1 to 3 days after confirmation.
settlement_records["settlement_date"] = (
    pd.to_datetime(settlement_records["confirmation_date"])
    + pd.to_timedelta(
        np.random.choice([1, 2, 3], size=len(settlement_records)),
        unit="D"
    )
)

# Calculate settlement amount using confirmed quantity and confirmed price.
settlement_records["settlement_amount"] = (
    settlement_records["confirmed_quantity"] * settlement_records["confirmed_price"]
).round(2)

# Simulate settlement status.
# SETTLED is intentionally more frequent than PENDING or FAILED.
settlement_records["settlement_status"] = np.random.choice(
    ["SETTLED", "SETTLED", "SETTLED", "PENDING", "FAILED"],
    size=len(settlement_records)
)

# Select a small number of trades to receive delayed settlement dates.
delay_ids = settlement_records.sample(5, random_state=5)["trade_id"].tolist()

# Add five extra days to selected settlement dates to simulate settlement delays.
settlement_records.loc[
    settlement_records["trade_id"].isin(delay_ids),
    "settlement_date"
] = (
    pd.to_datetime(settlement_records["settlement_date"])
    + pd.to_timedelta(5, unit="D")
)

# Save settlement records for downstream checks.
settlement_records.to_csv(TRADE_OPS_DIR / "settlement_records.csv", index=False)

print("Settlement records generated:", settlement_records.shape)
settlement_records.head()


Settlement records generated: (115, 8)


,trade_id,confirmation_date,asset,confirmed_quantity,confirmed_price,settlement_date,settlement_amount,settlement_status
0,T00001,2021-03-13,AUDUSD,11200,0.77872,2021-03-16,8721.66,SETTLED
1,T00002,2021-03-18,SP500,18900,3973.52103,2021-03-19,75099547.47,FAILED
2,T00003,2021-03-29,AUDUSD,10300,0.76360,2021-03-30,7865.08,SETTLED
3,T00004,2021-04-14,NASDAQ100,2400,13801.65789,2021-04-16,33123978.94,SETTLED
4,T00005,2021-04-27,GOLD,11600,1779.17251,2021-04-28,20638401.12,PENDING


## 6. Reload generated datasets and validate basic counts

This step reloads the three trade operations datasets from CSV.


In [6]:
# Reload exported datasets to simulate a clean downstream reconciliation process.
internal_trades = pd.read_csv(TRADE_OPS_DIR / "internal_trades.csv")
broker_confirmations = pd.read_csv(TRADE_OPS_DIR / "broker_confirmations.csv")
settlement_records = pd.read_csv(TRADE_OPS_DIR / "settlement_records.csv")

# Convert date columns back into datetime format after reading from CSV.
internal_trades["trade_date"] = pd.to_datetime(internal_trades["trade_date"])
broker_confirmations["confirmation_date"] = pd.to_datetime(broker_confirmations["confirmation_date"])
settlement_records["settlement_date"] = pd.to_datetime(settlement_records["settlement_date"])

# Basic sanity checks.
print("Internal trades:", internal_trades.shape)
print("Broker confirmations:", broker_confirmations.shape)
print("Settlement records:", settlement_records.shape)

# Broker confirmations should be fewer than internal trades because missing confirmations were simulated.
print("Missing confirmation count:", len(internal_trades) - len(broker_confirmations))


Internal trades: (120, 12)
Broker confirmations: (115, 7)
Settlement records: (115, 8)
Missing confirmation count: 5


## 7. Reconcile internal trades against broker confirmations

This step performs the core trade confirmation reconciliation.


In [7]:
# Join internal records with broker confirmations using trade_id and asset.
# Left join keeps every internal trade, including trades with missing confirmations.
trade_recon = internal_trades.merge(
    broker_confirmations,
    on=["trade_id", "asset"],
    how="left"
)

print("Trade reconciliation table:", trade_recon.shape)
trade_recon.head()


Trade reconciliation table: (120, 17)


,trade_id,trade_date,asset,side,quantity,execution_price,strategy_return_after_cost,total_cost,trader,strategy,notional,expected_pnl,confirmation_date,confirmed_quantity,confirmed_price,reported_pnl,broker_status
0,T00001,2021-03-12,AUDUSD,BUY,11200,0.77876,0.006359,0.000834,Desk_A,MA_Crossover,8722.11,48.19,2021-03-13,11200.0,0.77872,49.59,CONFIRMED
1,T00002,2021-03-17,SP500,BUY,18900,3974.12012,0.001456,0.001424,Desk_C,MA_Crossover,75110870.27,2409.66,2021-03-18,18900.0,3973.52103,2406.55,CONFIRMED
2,T00003,2021-03-29,AUDUSD,SELL,10200,0.76374,-0.000739,0.000739,Desk_C,MA_Crossover,7790.15,-11.51,2021-03-29,10300.0,0.76360,237.45,CONFIRMED
3,T00004,2021-04-14,NASDAQ100,BUY,2400,13803.91016,-0.014657,0.001603,Desk_B,MA_Crossover,33129384.38,-538662.40,2021-04-14,2400.0,13801.65789,-538664.87,CONFIRMED
4,T00005,2021-04-26,GOLD,BUY,11600,1779.19995,0.000103,0.001135,Desk_A,MA_Crossover,20638719.42,-21282.93,2021-04-27,11600.0,1779.17251,-21285.88,CONFIRMED


## 8. Apply tolerance thresholds for price and PnL checks

Not every small difference should become an exception. In operations control, tolerance thresholds help distinguish normal rounding/noise from material breaks.

This module uses:

- price discrepancy threshold: 0.2%;
- absolute PnL variance threshold: 100.


In [8]:
# Define tolerance thresholds used for exception monitoring.
PRICE_TOLERANCE_RATE = 0.002   # 0.2% price difference threshold
PNL_TOLERANCE_ABS = 100        # absolute PnL variance threshold

# Calculate absolute price difference rate.
trade_recon["price_diff_rate"] = (
    (trade_recon["confirmed_price"] - trade_recon["execution_price"]).abs()
    / trade_recon["execution_price"]
)

# Calculate difference between broker-reported PnL and internally expected PnL.
trade_recon["pnl_diff"] = (
    trade_recon["reported_pnl"] - trade_recon["expected_pnl"]
)

trade_recon[[
    "trade_id", "asset", "execution_price", "confirmed_price",
    "price_diff_rate", "expected_pnl", "reported_pnl", "pnl_diff"
]].head()


,trade_id,asset,execution_price,confirmed_price,price_diff_rate,expected_pnl,reported_pnl,pnl_diff
0,T00001,AUDUSD,0.77876,0.77872,0.000051,48.19,49.59,1.40
1,T00002,SP500,3974.12012,3973.52103,0.000151,2409.66,2406.55,-3.11
2,T00003,AUDUSD,0.76374,0.76360,0.000183,-11.51,237.45,248.96
3,T00004,NASDAQ100,13803.91016,13801.65789,0.000163,-538662.40,-538664.87,-2.47
4,T00005,GOLD,1779.19995,1779.17251,0.000015,-21282.93,-21285.88,-2.95


## 9. Generate trade exception report

This step flags trade-level exceptions across broker confirmation controls.

The output file `trade_exception_report.csv` is the main exception report for trade operations review.


In [9]:
# Store all detected exceptions in a list of dictionaries.
exceptions = []

# Iterate through each reconciled trade and apply exception rules.
for _, row in trade_recon.iterrows():
    trade_id = row["trade_id"]

    # Rule 1: internal trade has no matching broker confirmation.
    if pd.isna(row["confirmed_quantity"]):
        exceptions.append({
            "trade_id": trade_id,
            "asset": row["asset"],
            "exception_type": "Missing Broker Confirmation",
            "severity": "High",
            "description": "Internal trade record has no matching broker confirmation."
        })
        # If confirmation is missing, skip other broker-side checks for this trade.
        continue

    # Rule 2: confirmed quantity differs from internal quantity.
    if row["quantity"] != row["confirmed_quantity"]:
        exceptions.append({
            "trade_id": trade_id,
            "asset": row["asset"],
            "exception_type": "Quantity Mismatch",
            "severity": "High",
            "description": (
                f"Internal quantity {row['quantity']} differs from "
                f"confirmed quantity {row['confirmed_quantity']}."
            )
        })

    # Rule 3: confirmed price differs from execution price beyond tolerance.
    if row["price_diff_rate"] > PRICE_TOLERANCE_RATE:
        exceptions.append({
            "trade_id": trade_id,
            "asset": row["asset"],
            "exception_type": "Price Discrepancy",
            "severity": "Medium",
            "description": (
                f"Execution price and confirmed price differ by "
                f"{row['price_diff_rate']:.2%}, exceeding tolerance."
            )
        })

    # Rule 4: broker-reported PnL differs from expected PnL beyond tolerance.
    if abs(row["pnl_diff"]) > PNL_TOLERANCE_ABS:
        exceptions.append({
            "trade_id": trade_id,
            "asset": row["asset"],
            "exception_type": "PnL Variance",
            "severity": "Medium",
            "description": (
                f"Expected PnL and reported PnL differ by "
                f"{row['pnl_diff']:.2f}."
            )
        })

# Convert exception records into a structured DataFrame.
trade_exception_report = pd.DataFrame(exceptions)

# Save the exception report for GitHub, SQL review, or Tableau.
trade_exception_report.to_csv(
    OUTPUT_DIR / "trade_exception_report.csv",
    index=False
)

print("Trade exceptions detected:", trade_exception_report.shape)
trade_exception_report.head(10)


Trade exceptions detected: (22, 5)


,trade_id,asset,exception_type,severity,description
0,T00003,AUDUSD,Quantity Mismatch,High,Internal quantity 10200 differs from confirmed...
1,T00003,AUDUSD,PnL Variance,Medium,Expected PnL and reported PnL differ by 248.96.
2,T00006,EURUSD,Price Discrepancy,Medium,Execution price and confirmed price differ by ...
3,T00007,AUDUSD,Price Discrepancy,Medium,Execution price and confirmed price differ by ...
4,T00014,EURUSD,PnL Variance,Medium,Expected PnL and reported PnL differ by 247.20.
5,T00017,AUDUSD,PnL Variance,Medium,Expected PnL and reported PnL differ by 249.90.
6,T00020,NASDAQ100,PnL Variance,Medium,Expected PnL and reported PnL differ by 247.04.
7,T00025,AUDUSD,Quantity Mismatch,High,Internal quantity 19700 differs from confirmed...
8,T00026,GOLD,PnL Variance,Medium,Expected PnL and reported PnL differ by 254.07.
9,T00042,NASDAQ100,Quantity Mismatch,High,Internal quantity 6400 differs from confirmed ...


## 10. Reconcile settlement records and detect settlement exceptions

This step checks whether confirmed/internal trades have valid settlement records.

The settlement control flags:

- missing settlement records;
- failed or pending settlement status;
- settlement delays longer than four days.


In [10]:
# Join settlement records onto the trade reconciliation table.
settlement_recon = trade_recon.merge(
    settlement_records[[
        "trade_id", "settlement_date",
        "settlement_status", "settlement_amount"
    ]],
    on="trade_id",
    how="left"
)

# Calculate number of days from trade date to settlement date.
settlement_recon["settlement_lag_days"] = (
    settlement_recon["settlement_date"] - settlement_recon["trade_date"]
).dt.days

# Store all settlement exceptions here.
settlement_exceptions = []

# Apply settlement exception rules.
for _, row in settlement_recon.iterrows():

    # Rule 1: no settlement record exists.
    if pd.isna(row["settlement_date"]):
        settlement_exceptions.append({
            "trade_id": row["trade_id"],
            "asset": row["asset"],
            "exception_type": "Missing Settlement Record",
            "severity": "High",
            "description": "Confirmed or internal trade has no settlement record."
        })
        continue

    # Rule 2: settlement is failed or still pending.
    if row["settlement_status"] in ["FAILED", "PENDING"]:
        settlement_exceptions.append({
            "trade_id": row["trade_id"],
            "asset": row["asset"],
            "exception_type": "Settlement Not Completed",
            "severity": "High",
            "description": f"Settlement status is {row['settlement_status']}."
        })

    # Rule 3: settlement lag is longer than the operational threshold.
    if row["settlement_lag_days"] > 4:
        settlement_exceptions.append({
            "trade_id": row["trade_id"],
            "asset": row["asset"],
            "exception_type": "Settlement Delay",
            "severity": "Medium",
            "description": f"Settlement lag is {row['settlement_lag_days']} days."
        })

# Convert settlement exceptions into a structured DataFrame.
settlement_exception_report = pd.DataFrame(settlement_exceptions)

# Save settlement exception output.
settlement_exception_report.to_csv(
    OUTPUT_DIR / "settlement_exception_report.csv",
    index=False
)

print("Settlement exceptions detected:", settlement_exception_report.shape)
settlement_exception_report.head(10)


Settlement exceptions detected: (54, 5)


,trade_id,asset,exception_type,severity,description
0,T00002,SP500,Settlement Not Completed,High,Settlement status is FAILED.
1,T00005,GOLD,Settlement Not Completed,High,Settlement status is PENDING.
2,T00009,AUDUSD,Settlement Not Completed,High,Settlement status is FAILED.
3,T00010,NASDAQ100,Settlement Not Completed,High,Settlement status is PENDING.
4,T00012,GOLD,Settlement Not Completed,High,Settlement status is PENDING.
5,T00014,EURUSD,Settlement Not Completed,High,Settlement status is FAILED.
6,T00018,EURUSD,Settlement Delay,Medium,Settlement lag is 7.0 days.
7,T00020,NASDAQ100,Settlement Not Completed,High,Settlement status is PENDING.
8,T00021,AUDUSD,Settlement Not Completed,High,Settlement status is FAILED.
9,T00024,NASDAQ100,Settlement Delay,Medium,Settlement lag is 7.0 days.


## 11. Summarise exceptions by type and severity

This step aggregates exception reports for management-style monitoring.

These summary tables are useful for README reporting, dashboard design, and interview discussion because they show what types of breaks occurred and how severe they were.


In [11]:
# Summarise trade confirmation exceptions by type and severity.
trade_reconciliation_summary = (
    trade_exception_report
    .groupby(["exception_type", "severity"])
    .size()
    .reset_index(name="exception_count")
    .sort_values("exception_count", ascending=False)
)

# Summarise settlement exceptions by type and severity.
settlement_exception_summary = (
    settlement_exception_report
    .groupby(["exception_type", "severity"])
    .size()
    .reset_index(name="exception_count")
    .sort_values("exception_count", ascending=False)
)

# Save summary outputs.
trade_reconciliation_summary.to_csv(
    OUTPUT_DIR / "trade_reconciliation_summary.csv",
    index=False
)

settlement_exception_summary.to_csv(
    OUTPUT_DIR / "settlement_exception_summary.csv",
    index=False
)

display(trade_reconciliation_summary)
display(settlement_exception_summary)


,exception_type,severity,exception_count
1,PnL Variance,Medium,6
3,Quantity Mismatch,High,6
0,Missing Broker Confirmation,High,5
2,Price Discrepancy,Medium,5


,exception_type,severity,exception_count
2,Settlement Not Completed,High,44
0,Missing Settlement Record,High,5
1,Settlement Delay,Medium,5


## 12. Create asset-level exception summary

This additional output makes the module more useful for Tableau and stakeholder reporting.

It shows whether some instruments have more trade operations issues than others.


In [12]:
# Combine trade and settlement exception reports into one monitoring table.
all_exceptions = pd.concat(
    [
        trade_exception_report.assign(exception_stage="Trade Confirmation"),
        settlement_exception_report.assign(exception_stage="Settlement")
    ],
    ignore_index=True
)

# Aggregate exception counts by asset and exception stage.
asset_exception_summary = (
    all_exceptions
    .groupby(["asset", "exception_stage", "exception_type", "severity"])
    .size()
    .reset_index(name="exception_count")
    .sort_values(["asset", "exception_count"], ascending=[True, False])
)

# Save the asset-level summary for Tableau/dashboard use.
asset_exception_summary.to_csv(
    OUTPUT_DIR / "asset_exception_summary.csv",
    index=False
)

asset_exception_summary.head(10)


,asset,exception_stage,exception_type,severity,exception_count
1,AUDUSD,Settlement,Settlement Not Completed,High,11
5,AUDUSD,Trade Confirmation,Quantity Mismatch,High,3
0,AUDUSD,Settlement,Missing Settlement Record,High,2
2,AUDUSD,Trade Confirmation,Missing Broker Confirmation,High,2
3,AUDUSD,Trade Confirmation,PnL Variance,Medium,2
4,AUDUSD,Trade Confirmation,Price Discrepancy,Medium,1
8,EURUSD,Settlement,Settlement Not Completed,High,8
7,EURUSD,Settlement,Settlement Delay,Medium,2
11,EURUSD,Trade Confirmation,Price Discrepancy,Medium,2
6,EURUSD,Settlement,Missing Settlement Record,High,1


## 13. Create a trade operations scorecard

This step creates a compact operational scorecard.

The scorecard gives a quick view of:

- total internal trades;
- confirmation coverage;
- exception counts;
- exception rates;
- settlement completion performance.


In [13]:
# Calculate high-level monitoring metrics.
total_internal_trades = len(internal_trades)
total_confirmed_trades = len(broker_confirmations)
total_settlement_records = len(settlement_records)

total_trade_exceptions = len(trade_exception_report)
total_settlement_exceptions = len(settlement_exception_report)

# Count completed settlements.
settled_count = (settlement_records["settlement_status"] == "SETTLED").sum()

# Build a scorecard table for dashboard and README reporting.
scorecard = pd.DataFrame({
    "metric": [
        "Total Internal Trades",
        "Total Broker Confirmations",
        "Total Settlement Records",
        "Trade Exception Count",
        "Settlement Exception Count",
        "Broker Confirmation Coverage",
        "Settlement Completion Rate",
        "Trade Exception Rate",
        "Settlement Exception Rate"
    ],
    "value": [
        total_internal_trades,
        total_confirmed_trades,
        total_settlement_records,
        total_trade_exceptions,
        total_settlement_exceptions,
        round(total_confirmed_trades / total_internal_trades, 4),
        round(settled_count / total_settlement_records, 4),
        round(total_trade_exceptions / total_internal_trades, 4),
        round(total_settlement_exceptions / total_internal_trades, 4)
    ]
})

# Save the scorecard output.
scorecard.to_csv(
    OUTPUT_DIR / "trade_operations_scorecard.csv",
    index=False
)

scorecard


,metric,value
0,Total Internal Trades,120.0000
1,Total Broker Confirmations,115.0000
2,Total Settlement Records,115.0000
3,Trade Exception Count,22.0000
4,Settlement Exception Count,54.0000
5,Broker Confirmation Coverage,0.9583
6,Settlement Completion Rate,0.6174
7,Trade Exception Rate,0.1833
8,Settlement Exception Rate,0.4500


## 14. Export a final output manifest

This final step creates a small manifest listing all outputs produced by the module.

In [14]:
# Create a simple output manifest for documentation and review.
output_manifest = pd.DataFrame({
    "file_name": [
        "internal_trades.csv",
        "broker_confirmations.csv",
        "settlement_records.csv",
        "trade_exception_report.csv",
        "settlement_exception_report.csv",
        "trade_reconciliation_summary.csv",
        "settlement_exception_summary.csv",
        "asset_exception_summary.csv",
        "trade_operations_scorecard.csv"
    ],
    "location": [
        str(TRADE_OPS_DIR),
        str(TRADE_OPS_DIR),
        str(TRADE_OPS_DIR),
        str(OUTPUT_DIR),
        str(OUTPUT_DIR),
        str(OUTPUT_DIR),
        str(OUTPUT_DIR),
        str(OUTPUT_DIR),
        str(OUTPUT_DIR)
    ],
    "purpose": [
        "Internal trade blotter generated from strategy trade events.",
        "External broker confirmations with controlled simulated breaks.",
        "Settlement records with simulated failed, pending, and delayed settlements.",
        "Detailed trade confirmation exception report.",
        "Detailed settlement exception report.",
        "Aggregated trade exception summary by type and severity.",
        "Aggregated settlement exception summary by type and severity.",
        "Asset-level exception summary for dashboard reporting.",
        "High-level trade operations monitoring scorecard."
    ]
})

# Save manifest.
output_manifest.to_csv(
    OUTPUT_DIR / "trade_operations_output_manifest.csv",
    index=False
)

output_manifest


,file_name,location,purpose
0,internal_trades.csv,../data/trade_operations,Internal trade blotter generated from strategy...
1,broker_confirmations.csv,../data/trade_operations,External broker confirmations with controlled ...
2,settlement_records.csv,../data/trade_operations,"Settlement records with simulated failed, pend..."
3,trade_exception_report.csv,../trade_operations_outputs,Detailed trade confirmation exception report.
4,settlement_exception_report.csv,../trade_operations_outputs,Detailed settlement exception report.
5,trade_reconciliation_summary.csv,../trade_operations_outputs,Aggregated trade exception summary by type and...
6,settlement_exception_summary.csv,../trade_operations_outputs,Aggregated settlement exception summary by typ...
7,asset_exception_summary.csv,../trade_operations_outputs,Asset-level exception summary for dashboard re...
8,trade_operations_scorecard.csv,../trade_operations_outputs,High-level trade operations monitoring scorecard.
